# AI-Powered Resume Screening Assistant
### LangChain Agent-Based System for Intelligent Candidate Evaluation

---

## Overview

This notebook implements a **Resume Screening Assistant** that uses LangChain agents and tool-based workflows to:

- Parse resumes from PDF or plain text
- Accept and parse job descriptions
- Semantically match candidates to job requirements
- Score, rank, and produce structured evaluation reports

---

## User Guide

### Setup Instructions

**1. Install required dependencies:**
```bash
pip install langchain langchain-groq langchain-community langchain-huggingface pypdf2 python-docx faiss-cpu sentence-transformers pandas tabulate
```

**2. Set your Groq API key** in Cell 2 or as an environment variable:
```bash
export GROQ_API_KEY="gsk_..."
```

**3. Provide resumes** — either upload PDF files or paste text directly in the demo section.

### How to Run
- Run all cells top to bottom (`Kernel → Restart & Run All`)
- In the final demo cell, replace example resumes and job description with your own data

### Required Dependencies
| Package | Purpose |
|---|---|
| `langchain` | Agent framework and tool orchestration |
| `langchain-groq` | Groq LLM integration (fast, free models) |
| `langchain-huggingface` | Free embeddings via HuggingFace |
| `sentence-transformers` | Local embedding models |
| `PyPDF2` | PDF text extraction |
| `python-docx` | DOCX file processing |
| `faiss-cpu` | Semantic similarity via vector embeddings |
| `pandas` | Tabular result formatting |
| `tabulate` | Pretty-print tables |


---
## Cell 1 — Install Dependencies

In [82]:
!pip install langchain langchain-groq langchain-community langchain-huggingface pypdf2 python-docx faiss-cpu sentence-transformers pandas tabulate

Defaulting to user installation because normal site-packages is not writeable


---
## Cell 2 — Configuration & API Key

In [ ]:
import os

# ── Set your Groq API key here ────────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = "API_KEY_HERE"
# ─────────────────────────────────────────────────────────────────────────────

# llama-3.1-8b-instant: ~20K TPM free tier — much higher than llama-3.3-70b-versatile (12K)
LLM_MODEL        = "llama-3.1-8b-instant"
EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
TEMPERATURE      = 0.0

WEIGHT_SEMANTIC    = 0.40
WEIGHT_SKILLS      = 0.35
WEIGHT_EXPERIENCE  = 0.25

print(f"Using model : {LLM_MODEL}")
print(f"Scoring weights — semantic:{WEIGHT_SEMANTIC}  skills:{WEIGHT_SKILLS}  experience:{WEIGHT_EXPERIENCE}")


Using model : llama-3.1-8b-instant
Scoring weights — semantic:0.4  skills:0.35  experience:0.25


---
## Cell 3 — Imports

In [84]:
import io
import re
import json
import warnings
import textwrap
from pathlib import Path
from typing import List, Optional, Dict, Any

import pandas as pd
warnings.filterwarnings("ignore")

# LangChain core — compatible with LangChain 1.x
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Pydantic for structured output validation
from pydantic import BaseModel, Field

# PDF / DOCX extraction
try:
    import PyPDF2
except ImportError:
    PyPDF2 = None

try:
    import docx
except ImportError:
    docx = None

print("All imports successful.")


All imports successful.


---
## Cell 4 — Pydantic Output Schemas

Structured models guarantee consistent, machine-readable output from every LLM call.

In [85]:
class ParsedResume(BaseModel):
    """Structured representation of a single parsed resume."""
    candidate_name:     str            = Field(description="Full name of the candidate")
    email:              Optional[str]  = Field(None, description="Contact email address")
    phone:              Optional[str]  = Field(None, description="Phone number")
    years_experience:   float          = Field(0.0,  description="Total years of professional experience")
    experience_level:   str            = Field(description="One of: Junior, Mid-Level, Senior, Lead, Executive")
    current_role:       Optional[str]  = Field(None, description="Most recent job title")
    education:          List[str]      = Field(default_factory=list, description="Degrees / certifications")
    skills:             List[str]      = Field(default_factory=list, description="Technical and soft skills listed")
    skills_categories:  Dict[str,List[str]] = Field(
        default_factory=dict,
        description="Skills grouped by category, e.g. {'Programming': ['Python'], 'Cloud': ['AWS']}"
    )
    work_history:       List[str]      = Field(default_factory=list, description="Previous roles / companies")
    raw_text:           str            = Field(default="", description="Original extracted text", exclude=True)


class ParsedJobDescription(BaseModel):
    """Structured representation of a job description."""
    job_title:              str            = Field(description="Title of the position")
    company:                Optional[str]  = Field(None, description="Hiring company name")
    required_skills:        List[str]      = Field(default_factory=list)
    preferred_skills:       List[str]      = Field(default_factory=list)
    min_years_experience:   float          = Field(0.0)
    required_education:     Optional[str]  = Field(None)
    responsibilities:       List[str]      = Field(default_factory=list)
    keywords:               List[str]      = Field(default_factory=list, description="Domain keywords for semantic search")


class CandidateEvaluation(BaseModel):
    """Full evaluation result for one candidate against one job description."""
    candidate_name:         str
    job_title:              str
    semantic_score:         float  = Field(ge=0.0, le=1.0, description="Vector-space similarity [0,1]")
    skills_match_score:     float  = Field(ge=0.0, le=1.0)
    experience_score:       float  = Field(ge=0.0, le=1.0)
    overall_score:          float  = Field(ge=0.0, le=100.0, description="Weighted composite score out of 100")
    matched_skills:         List[str]
    missing_skills:         List[str]
    skill_gaps:             List[str]
    strengths:              List[str]
    experience_level:       str
    years_experience:       float
    recommendation:         str    = Field(description="One of: Strong Hire, Hire, Maybe, No Hire")
    llm_summary:            str    = Field(description="Narrative evaluation paragraph")


print("Output schemas defined.")

Output schemas defined.


---
## Cell 5 — Document Processing Utilities

Handles PDF, DOCX, and plain-text extraction with robust error handling.

In [86]:
class DocumentProcessor:
    """
    Extracts plain text from PDF, DOCX, or TXT files.
    Falls back gracefully when a library is unavailable.
    """

    @staticmethod
    def extract_from_pdf(source) -> str:
        """
        Parameters
        ----------
        source : str | Path | bytes
            File path, Path object, or raw PDF bytes.
        """
        if PyPDF2 is None:
            raise ImportError("PyPDF2 is not installed. Run: pip install PyPDF2")

        try:
            if isinstance(source, (str, Path)):
                file_obj = open(source, "rb")
                should_close = True
            else:
                file_obj = io.BytesIO(source)
                should_close = False

            reader = PyPDF2.PdfReader(file_obj)
            pages  = [page.extract_text() or "" for page in reader.pages]
            text   = "\n".join(pages).strip()

            if should_close:
                file_obj.close()

            if not text:
                raise ValueError("PDF appears to be empty or image-based (no extractable text).")
            return text

        except Exception as exc:
            raise ValueError(f"PDF extraction failed: {exc}") from exc

    @staticmethod
    def extract_from_docx(path) -> str:
        if docx is None:
            raise ImportError("python-docx is not installed. Run: pip install python-docx")
        try:
            document = docx.Document(str(path))
            return "\n".join(p.text for p in document.paragraphs if p.text.strip())
        except Exception as exc:
            raise ValueError(f"DOCX extraction failed: {exc}") from exc

    @staticmethod
    def extract_from_text(path) -> str:
        try:
            return Path(path).read_text(encoding="utf-8", errors="ignore").strip()
        except Exception as exc:
            raise ValueError(f"Text file read failed: {exc}") from exc

    @classmethod
    def extract(cls, source) -> str:
        """
        Auto-detect file type and extract text.
        ``source`` may be a file path (str/Path) or raw text string.
        """
        if isinstance(source, str) and not Path(source).exists():
            # Treat as raw text
            return source.strip()

        path = Path(source)
        suffix = path.suffix.lower()

        if suffix == ".pdf":
            return cls.extract_from_pdf(path)
        elif suffix in (".docx", ".doc"):
            return cls.extract_from_docx(path)
        elif suffix in (".txt", ".md"):
            return cls.extract_from_text(path)
        else:
            # Unknown extension — try plain text
            try:
                return cls.extract_from_text(path)
            except Exception:
                raise ValueError(f"Unsupported file type: {suffix}. Supported: PDF, DOCX, TXT")


print("DocumentProcessor ready.")

DocumentProcessor ready.


---
## Cell 6 — LLM Parser: Resume & Job Description

Uses structured LLM calls with JSON output to parse raw text into validated Pydantic models.

In [87]:
class LLMParser:
    """
    Thin wrapper around ChatGroq that parses raw document text
    into structured Pydantic objects using JSON mode.
    """

    def __init__(self, model: str = LLM_MODEL, temperature: float = TEMPERATURE):
        self.llm = ChatGroq(
            model=model,
            temperature=temperature,
            api_key=os.environ.get("GROQ_API_KEY")
        )

    # ── Resume Parser ─────────────────────────────────────────────────────────
    def parse_resume(self, raw_text: str, candidate_id: str = "Unknown") -> ParsedResume:
        """Convert raw resume text into a ParsedResume object."""
        if not raw_text or not raw_text.strip():
            raise ValueError(f"Empty resume text for candidate '{candidate_id}'.")

        prompt = f"""
You are an expert HR data extractor. Parse the resume below and return ONLY valid JSON
matching this schema (no extra keys, no markdown):

{{
  "candidate_name":     string,
  "email":              string | null,
  "phone":              string | null,
  "years_experience":   float,
  "experience_level":   "Junior" | "Mid-Level" | "Senior" | "Lead" | "Executive",
  "current_role":       string | null,
  "education":          [string],
  "skills":             [string],
  "skills_categories":  {{ category: [skill] }},
  "work_history":       [string]
}}

Rules:
- Infer experience_level from years: <2=Junior, 2-5=Mid-Level, 5-10=Senior, 10+=Lead/Executive
- skills_categories groups skills, e.g. Programming, Cloud, Databases, Soft Skills
- work_history entries: "Role at Company (Year-Year)"
- If information is missing, use null / empty list / 0.0 as appropriate

RESUME TEXT:
---
{raw_text[:1500]}
---
"""
        response = self.llm.invoke(prompt)
        raw = response.content.strip()
        # Strip markdown code fences if the model wrapped the JSON
        if raw.startswith("```"):
            raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw)
            raw = re.sub(r"```$", "", raw).strip()
        data   = json.loads(raw)
        resume = ParsedResume(**data)
        resume.raw_text = raw_text
        return resume

    # ── Job Description Parser ─────────────────────────────────────────────────
    def parse_job_description(self, jd_text: str) -> ParsedJobDescription:
        """Convert raw job-description text into a ParsedJobDescription object."""
        if not jd_text or not jd_text.strip():
            raise ValueError("Job description text is empty.")

        prompt = f"""
You are an expert HR data extractor. Parse the job description below and return ONLY valid JSON
matching this schema:

{{
  "job_title":             string,
  "company":               string | null,
  "required_skills":       [string],
  "preferred_skills":      [string],
  "min_years_experience":  float,
  "required_education":    string | null,
  "responsibilities":      [string],
  "keywords":              [string]
}}

Rules:
- required_skills are explicitly listed as required/must-have
- preferred_skills are listed as nice-to-have/bonus
- keywords capture domain terms useful for semantic search

JOB DESCRIPTION:
---
{jd_text[:1200]}
---
"""
        response = self.llm.invoke(prompt)
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw)
            raw = re.sub(r"```$", "", raw).strip()
        data = json.loads(raw)
        return ParsedJobDescription(**data)


print("LLMParser ready.")

LLMParser ready.


---
## Cell 7 — Semantic Similarity Engine

Builds a FAISS vector store from resume chunks, then scores each resume against the job description using cosine similarity.

In [88]:
class SemanticMatcher:
    """
    Embeds resumes and job description into a shared vector space,
    then returns cosine-similarity scores in [0, 1].
    """

    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODEL,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        self.splitter   = RecursiveCharacterTextSplitter(
            chunk_size=800, chunk_overlap=100
        )

    def _embed_text(self, text: str) -> List[float]:
        return self.embeddings.embed_query(text)

    @staticmethod
    def _cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
        import math
        dot   = sum(a * b for a, b in zip(vec_a, vec_b))
        norm_a = math.sqrt(sum(a * a for a in vec_a))
        norm_b = math.sqrt(sum(b * b for b in vec_b))
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)

    def score(self, resume_text: str, jd_text: str) -> float:
        """
        Returns a similarity score in [0, 1].
        Uses the average similarity across resume chunks vs. the full JD embedding.
        """
        if not resume_text.strip() or not jd_text.strip():
            return 0.0

        jd_vec      = self._embed_text(jd_text[:3000])
        chunks      = self.splitter.split_text(resume_text)
        if not chunks:
            return 0.0

        scores = [
            self._cosine_similarity(self._embed_text(chunk), jd_vec)
            for chunk in chunks[:6]   # cap at 6 chunks to keep latency low
        ]
        return round(max(scores), 4)  # take the best-matching chunk


print("SemanticMatcher ready.")

SemanticMatcher ready.


---
## Cell 8 — Scoring Engine

Combines semantic similarity, hard-skill matching, and experience scoring into a single weighted composite score.

In [89]:
class ScoringEngine:
    """
    Computes three sub-scores and merges them into an overall [0, 100] score.
    """

    @staticmethod
    def skills_match(resume: ParsedResume, jd: ParsedJobDescription) -> Dict[str, Any]:
        """
        Case-insensitive keyword matching between candidate skills and JD skills.
        Returns score in [0,1] plus matched / missing / gap lists.
        """
        candidate_skills_lower = {s.lower() for s in resume.skills}
        required_lower  = {s.lower() for s in jd.required_skills}
        preferred_lower = {s.lower() for s in jd.preferred_skills}

        # Also do partial matching (substring)
        def fuzzy_contains(skill_set, query):
            return any(query in s or s in query for s in skill_set)

        matched_req  = [s for s in jd.required_skills  if fuzzy_contains(candidate_skills_lower, s.lower())]
        matched_pref = [s for s in jd.preferred_skills if fuzzy_contains(candidate_skills_lower, s.lower())]
        missing_req  = [s for s in jd.required_skills  if s not in matched_req]

        # Score: required skills are weighted 2×, preferred 1×
        total_weight   = len(jd.required_skills) * 2 + len(jd.preferred_skills)
        earned_weight  = len(matched_req) * 2 + len(matched_pref)
        score = (earned_weight / total_weight) if total_weight > 0 else 0.5

        return {
            "score":         min(round(score, 4), 1.0),
            "matched":       matched_req + matched_pref,
            "missing":       missing_req,
            "skill_gaps":    missing_req,  # alias for report
        }

    @staticmethod
    def experience_score(resume: ParsedResume, jd: ParsedJobDescription) -> float:
        """
        Scores experience relative to job requirement.
        - Exact or over-qualified:  1.0
        - 1 year short:             0.7
        - 2 years short:            0.4
        - 3+ years short:           0.1
        """
        if jd.min_years_experience <= 0:
            return 1.0  # No requirement specified

        gap = jd.min_years_experience - resume.years_experience
        if gap <= 0:
            return 1.0
        elif gap <= 1:
            return 0.7
        elif gap <= 2:
            return 0.4
        else:
            return 0.1

    @staticmethod
    def composite_score(semantic: float, skills: float, experience: float) -> float:
        """Weighted composite, returned as 0-100."""
        raw = (
            semantic   * WEIGHT_SEMANTIC   +
            skills     * WEIGHT_SKILLS     +
            experience * WEIGHT_EXPERIENCE
        )
        return round(raw * 100, 2)

    @staticmethod
    def recommendation(score: float) -> str:
        """Convert numeric score to hiring recommendation."""
        if score >= 80:
            return "Strong Hire"
        elif score >= 65:
            return "Hire"
        elif score >= 50:
            return "Maybe"
        else:
            return "No Hire"


print("ScoringEngine ready.")

ScoringEngine ready.


---
## Cell 9 — LangChain Tools

Each capability is exposed as a LangChain `@tool` so the agent can reason about which to call and in what order.

In [90]:
# ── Shared singletons (initialised once, reused by all tools) ─────────────────
_parser  = LLMParser()
_matcher = SemanticMatcher()
_scorer  = ScoringEngine()

# Global state: stored between tool calls within a single agent run
_state: Dict[str, Any] = {}


# ── Tool 1: Load & extract text from a resume file ────────────────────────────
@tool
def extract_resume_text(source: str) -> str:
    """
    Extract plain text from a resume.
    `source` can be a file path (PDF/DOCX/TXT) or a raw text string.
    Returns the extracted text or an error message.
    """
    try:
        text = DocumentProcessor.extract(source)
        return f"SUCCESS: Extracted {len(text)} characters.\n\nPREVIEW:\n{text[:500]}..."
    except Exception as e:
        return f"ERROR: {e}"


# ── Tool 2: Parse a resume into structured JSON ────────────────────────────────
@tool
def parse_resume(resume_text: str, candidate_id: str = "Candidate") -> str:
    """
    Parse raw resume text into a structured JSON profile.
    Returns a JSON string with name, skills, experience, education, etc.
    """
    try:
        parsed = _parser.parse_resume(resume_text, candidate_id)
        _state.setdefault("resumes", {})[parsed.candidate_name] = parsed
        data = parsed.model_dump()
        data.pop("raw_text", None)
        return json.dumps(data, indent=2)
    except Exception as e:
        return f"ERROR parsing resume: {e}"


# ── Tool 3: Parse job description ─────────────────────────────────────────────
@tool
def parse_job_description(jd_text: str) -> str:
    """
    Parse a job description into structured JSON.
    Returns job title, required/preferred skills, experience requirements, etc.
    """
    try:
        jd = _parser.parse_job_description(jd_text)
        _state["job_description"] = jd
        return json.dumps(jd.model_dump(), indent=2)
    except Exception as e:
        return f"ERROR parsing job description: {e}"


# ── Tool 4: Compute semantic similarity ───────────────────────────────────────
@tool
def compute_semantic_similarity(resume_text: str, jd_text: str) -> str:
    """
    Compute cosine similarity between a resume and a job description using embeddings.
    Returns a score between 0.0 (no match) and 1.0 (perfect match).
    """
    try:
        score = _matcher.score(resume_text, jd_text)
        return f"Semantic similarity score: {score:.4f} ({score*100:.1f}%)"
    except Exception as e:
        return f"ERROR computing similarity: {e}"


# ── Tool 5: Full candidate evaluation ────────────────────────────────────────
@tool
def evaluate_candidate(candidate_name: str, jd_text: str) -> str:
    """
    Perform a complete evaluation of a candidate (must be parsed first) against a job description.
    Returns a detailed JSON evaluation including score, matched skills, gaps, and recommendation.
    """
    try:
        resumes = _state.get("resumes", {})
        if candidate_name not in resumes:
            return f"ERROR: Candidate '{candidate_name}' not found. Parse their resume first."

        resume = resumes[candidate_name]
        # Reuse cached parsed JD to avoid redundant LLM calls and token waste
        jd = _state.get("job_description") or _parser.parse_job_description(jd_text[:1200])

        # Sub-scores
        sem_score   = _matcher.score(resume.raw_text, jd_text)
        skills_data = _scorer.skills_match(resume, jd)
        exp_score   = _scorer.experience_score(resume, jd)
        overall     = _scorer.composite_score(sem_score, skills_data["score"], exp_score)
        rec         = _scorer.recommendation(overall)

        # LLM narrative
        narrative_prompt = f"""
Write a concise 3-sentence HR evaluation for this candidate:
- Candidate: {resume.candidate_name} ({resume.years_experience} yrs exp, {resume.experience_level})
- Role: {jd.job_title}
- Overall score: {overall}/100 — {rec}
- Matched skills: {', '.join(skills_data['matched'][:5])}
- Missing skills: {', '.join(skills_data['missing'][:5])}
Be professional and objective."""

        llm_resp  = ChatGroq(
            model=LLM_MODEL,
            temperature=0,
            api_key=os.environ.get("GROQ_API_KEY")
        ).invoke(narrative_prompt)
        narrative = llm_resp.content.strip()

        evaluation = CandidateEvaluation(
            candidate_name     = resume.candidate_name,
            job_title          = jd.job_title,
            semantic_score     = sem_score,
            skills_match_score = skills_data["score"],
            experience_score   = exp_score,
            overall_score      = overall,
            matched_skills     = skills_data["matched"],
            missing_skills     = skills_data["missing"],
            skill_gaps         = skills_data["skill_gaps"],
            strengths          = resume.skills[:5],
            experience_level   = resume.experience_level,
            years_experience   = resume.years_experience,
            recommendation     = rec,
            llm_summary        = narrative,
        )

        _state.setdefault("evaluations", {})[candidate_name] = evaluation
        return json.dumps(evaluation.model_dump(), indent=2)

    except Exception as e:
        return f"ERROR evaluating candidate: {e}"


# ── Tool 6: Rank all evaluated candidates ─────────────────────────────────────
@tool
def rank_candidates(trigger: str = "run") -> str:
    """
    Rank all evaluated candidates by overall score (descending).
    Returns a sorted table with scores, recommendations, and key highlights.
    Call this after evaluating all candidates.
    """
    evaluations = _state.get("evaluations", {})
    if not evaluations:
        return "ERROR: No candidate evaluations found. Evaluate candidates first."

    rows = []
    for ev in sorted(evaluations.values(), key=lambda x: x.overall_score, reverse=True):
        rows.append({
            "Rank":            "",
            "Candidate":       ev.candidate_name,
            "Score":           f"{ev.overall_score:.1f}/100",
            "Semantic":        f"{ev.semantic_score*100:.1f}%",
            "Skills Match":    f"{ev.skills_match_score*100:.1f}%",
            "Experience":      f"{ev.years_experience} yrs ({ev.experience_level})",
            "Recommendation":  ev.recommendation,
            "Top Gaps":        ", ".join(ev.skill_gaps[:3]) or "None",
        })

    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"
    df.drop(columns=["Rank"], inplace=True)

    return df.to_string()


# ── Tool 7: Generate full screening report ────────────────────────────────────
@tool
def generate_screening_report(trigger: str = "run") -> str:
    """
    Generate a comprehensive hiring report covering all evaluated candidates.
    Includes executive summary, ranked table, per-candidate summaries, and hiring recommendations.
    Call this last, after all evaluations are complete.
    """
    evaluations = _state.get("evaluations", {})
    jd          = _state.get("job_description")

    if not evaluations:
        return "ERROR: No evaluations to report on."

    sorted_evals = sorted(evaluations.values(), key=lambda x: x.overall_score, reverse=True)
    top          = sorted_evals[0]
    jd_title     = jd.job_title if jd else "Position"

    lines = [
        "=" * 70,
        f"  RESUME SCREENING REPORT — {jd_title.upper()}",
        "=" * 70,
        f"  Total Candidates Evaluated : {len(sorted_evals)}",
        f"  Top Candidate              : {top.candidate_name} ({top.overall_score:.1f}/100)",
        f"  Recommended for Interview  : {sum(1 for e in sorted_evals if e.recommendation in ('Strong Hire','Hire'))}",
        "=" * 70,
        "",
        "RANKED CANDIDATES",
        "-" * 70,
    ]

    for rank, ev in enumerate(sorted_evals, 1):
        lines += [
            f"  #{rank}  {ev.candidate_name}",
            f"      Overall Score   : {ev.overall_score:.1f}/100  [{ev.recommendation}]",
            f"      Semantic Match  : {ev.semantic_score*100:.1f}%",
            f"      Skills Match    : {ev.skills_match_score*100:.1f}%",
            f"      Experience      : {ev.years_experience} yrs — {ev.experience_level}",
            f"      Matched Skills  : {', '.join(ev.matched_skills[:5]) or 'None'}",
            f"      Skill Gaps      : {', '.join(ev.skill_gaps[:5]) or 'None'}",
            f"      Evaluation      : {ev.llm_summary}",
            "-" * 70,
        ]

    lines += [
        "",
        "SCORING METHODOLOGY",
        f"  Semantic Similarity : {WEIGHT_SEMANTIC*100:.0f}%",
        f"  Skills Match        : {WEIGHT_SKILLS*100:.0f}%",
        f"  Experience Match    : {WEIGHT_EXPERIENCE*100:.0f}%",
        "  Thresholds: Strong Hire ≥80 | Hire ≥65 | Maybe ≥50 | No Hire <50",
        "=" * 70,
    ]

    report = "\n".join(lines)
    _state["final_report"] = report
    return report


# Collect all tools for the agent
TOOLS = [
    extract_resume_text,
    parse_resume,
    parse_job_description,
    compute_semantic_similarity,
    evaluate_candidate,
    rank_candidates,
    generate_screening_report,
]

print(f"{len(TOOLS)} tools registered for the agent.")

7 tools registered for the agent.


---
## Cell 10 — Build the LangChain Agent

Uses `create_openai_tools_agent` with Groq-backed LLM (fast, free models) and a custom system prompt that guides the agent through the screening workflow.

In [91]:
SYSTEM_PROMPT = """\
You are an expert AI Recruiter powered by LangChain tools.
Your task is to screen candidate resumes against a job description and produce
a ranked, structured hiring report.

WORKFLOW (follow this order):
1. Parse the job description using `parse_job_description`.
2. For each candidate resume:
   a. If given a file path, call `extract_resume_text` first.
   b. Call `parse_resume` to extract structured data.
   c. Call `evaluate_candidate` with the candidate name and full JD text.
3. After ALL candidates are evaluated, call `rank_candidates`.
4. Finally call `generate_screening_report` to produce the full report.

RULES:
- Never skip a candidate; evaluate every one before ranking.
- If a tool returns an ERROR, note it, attempt once to fix inputs, then continue.
- Always pass the FULL raw text to `parse_resume` and `evaluate_candidate`.
- Be objective and data-driven. Base recommendations solely on scores and gaps.
"""

llm = ChatGroq(
    model=LLM_MODEL,
    temperature=TEMPERATURE,
    api_key=os.environ.get("GROQ_API_KEY")
)

# LangChain 1.x: create_agent returns a CompiledStateGraph (LangGraph-backed)
agent_executor = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

print("Agent executor built and ready.")
print(f"Agent type: {type(agent_executor).__name__}")


Agent executor built and ready.
Agent type: CompiledStateGraph


---
## Cell 11 — Helper: Run Screening

A clean wrapper that resets state, invokes the agent, and prints the final report.

In [92]:
def run_screening(
    job_description: str,
    resumes: List[Dict[str, str]],
) -> str:
    """
    Direct pipeline — calls tools in sequence, no agent loop.
    """
    _state.clear()
    JD_CAP     = 1200
    RESUME_CAP = 1500
    jd_snippet = job_description[:JD_CAP]

    # ── Step 1: Parse job description ────────────────────────────────────────
    print('[1/3] Parsing job description...')
    jd_result = parse_job_description.invoke(jd_snippet)
    if jd_result.startswith('ERROR'):
        raise RuntimeError(f'JD parsing failed: {jd_result}')
    print(f'      Done — {_state["job_description"].job_title}')

    # ── Step 2: Parse + evaluate each candidate ───────────────────────────────
    print(f'[2/3] Evaluating {len(resumes)} candidate(s)...')
    for r in resumes:
        name = r['name']
        text = (r.get('text') or DocumentProcessor.extract(r['path']))[:RESUME_CAP]

        print(f'      Parsing   {name}...')
        parse_result = parse_resume.invoke({'resume_text': text, 'candidate_id': name})
        if parse_result.startswith('ERROR'):
            print(f'      WARNING: {parse_result} — skipping {name}')
            continue

        # Attach raw_text — LLM may have slightly normalised the name
        actual_name = name
        if name not in _state.get('resumes', {}):
            actual_name = list(_state.get('resumes', {}).keys())[-1]
        _state['resumes'][actual_name].raw_text = text

        print(f'      Evaluating {actual_name}...')
        eval_result = evaluate_candidate.invoke({'candidate_name': actual_name, 'jd_text': jd_snippet})
        if eval_result.startswith('ERROR'):
            print(f'      WARNING: {eval_result}')
        else:
            print(f'      Done:     {actual_name}')

    if not _state.get('evaluations'):
        raise RuntimeError('No candidates were successfully evaluated. Check errors above.')

    # ── Step 3: Rank + final report ───────────────────────────────────────────
    print('[3/3] Generating report...')
    rank_candidates.invoke('run')
    report = generate_screening_report.invoke('run')

    print('\n' + '=' * 70)
    print(report)
    return report


print('run_screening() ready.')


run_screening() ready.


---
## Cell 12 — Example Usage: Sample Data

Three fictional candidates apply for a **Senior ML Engineer** role. Replace with real resumes and a real JD.

In [93]:
# ── Job Description ───────────────────────────────────────────────────────────
JOB_DESCRIPTION = """
Company: DataDriven Inc.
Position: Senior Machine Learning Engineer

About the Role:
We are seeking a Senior ML Engineer to design, build, and deploy production ML systems
at scale. You will collaborate with data scientists, product teams, and infrastructure
engineers to bring models from prototype to production.

Required Skills:
- Python (5+ years)
- PyTorch or TensorFlow
- MLflow or Kubeflow for experiment tracking and model registry
- Docker and Kubernetes
- AWS or GCP cloud platform experience
- REST API development (FastAPI or Flask)
- SQL and data pipeline experience

Preferred Skills:
- Distributed training (Horovod, DeepSpeed)
- Feature stores (Feast)
- Experience with LLMs and transformer architectures
- Spark or Dask for large-scale data processing
- CI/CD pipelines (GitHub Actions, Jenkins)

Requirements:
- Minimum 5 years of experience in ML engineering or related field
- Bachelor's or Master's in Computer Science, Data Science, or related field
- Proven track record of deploying ML models to production

Responsibilities:
- Design and implement end-to-end ML pipelines
- Optimize model performance and inference latency
- Mentor junior engineers
- Collaborate with stakeholders to define ML requirements
"""

# ── Candidate Resumes ─────────────────────────────────────────────────────────
RESUMES = [
    {
        "name": "Alice Chen",
        "text": """
        Alice Chen | alice.chen@email.com | +1-415-555-0198
        San Francisco, CA

        SUMMARY
        Machine Learning Engineer with 7 years of experience building and deploying
        large-scale ML systems. Specialized in NLP and transformer-based architectures.

        EXPERIENCE
        Senior ML Engineer — Stripe (2021–Present)
        - Led deployment of fraud-detection model serving 50M+ daily requests
        - Reduced inference latency by 40% via ONNX quantization and TensorRT
        - Built feature pipeline using Feast + Apache Spark on GCP
        - Mentored team of 4 junior engineers

        ML Engineer — Lyft (2018–2021)
        - Built demand forecasting models using PyTorch and deployed via Kubeflow
        - Developed REST APIs with FastAPI for real-time model serving
        - Managed AWS SageMaker pipelines and S3 data lake

        Data Scientist — StartupAI (2016–2018)
        - Trained NLP classifiers for customer intent detection
        - Maintained MLflow experiment registry

        EDUCATION
        M.S. Computer Science — Stanford University (2016)
        B.S. Mathematics — UC Berkeley (2014)

        SKILLS
        Python, PyTorch, TensorFlow, Kubeflow, MLflow, Docker, Kubernetes, FastAPI,
        AWS, GCP, Apache Spark, Feast, SQL, Git, CI/CD, Distributed Training (Horovod),
        LLMs, Transformers, ONNX, TensorRT
        """
    },
    {
        "name": "Bob Martinez",
        "text": """
        Bob Martinez | bob.m@techmail.com | +1-212-555-0374
        New York, NY

        PROFESSIONAL SUMMARY
        Software engineer transitioning into ML engineering. 3 years of backend
        development experience with 1 year focused on ML model deployment.

        WORK HISTORY
        Junior ML Engineer — FinTechCorp (2022–Present)
        - Assisted in deploying scikit-learn classification models via Flask APIs
        - Wrote SQL queries for training data extraction
        - Packaged models with Docker

        Backend Developer — WebAgency (2020–2022)
        - Built REST APIs in Python/Django for e-commerce platform
        - Maintained PostgreSQL databases

        EDUCATION
        B.S. Computer Science — NYU (2020)

        SKILLS
        Python, Flask, Django, Docker, SQL, PostgreSQL, scikit-learn, REST APIs,
        Git, Linux, Pandas, NumPy
        """
    },
    {
        "name": "Priya Sharma",
        "text": """
        Priya Sharma | priya.sharma@ml.io | +44-7700-900456
        London, UK

        PROFILE
        ML Engineer with 5 years of experience in NLP, recommendation systems, and
        cloud-native ML infrastructure. Delivered production models at two unicorn startups.

        EXPERIENCE
        ML Engineer — Monzo Bank (2021–Present)
        - Built real-time transaction categorisation model using PyTorch + TorchServe
        - Designed Kubernetes-based model serving infrastructure on AWS EKS
        - Implemented CI/CD for ML models using GitHub Actions
        - Created data pipelines with Apache Airflow and dbt

        ML Engineer — Babylon Health (2019–2021)
        - Fine-tuned BERT models for medical NLP tasks
        - Used MLflow for experiment tracking
        - Deployed models via FastAPI on GCP Cloud Run

        EDUCATION
        M.Eng. Electrical & Electronic Engineering — Imperial College London (2019)

        SKILLS
        Python, PyTorch, TensorFlow, FastAPI, Docker, Kubernetes, AWS, GCP,
        MLflow, SQL, Apache Airflow, dbt, BERT, Transformers, LLMs, GitHub Actions,
        TorchServe, REST APIs
        """
    }
]

print(f"Loaded: {len(RESUMES)} candidate resumes and 1 job description.")
print("Candidates:", [r['name'] for r in RESUMES])

Loaded: 3 candidate resumes and 1 job description.
Candidates: ['Alice Chen', 'Bob Martinez', 'Priya Sharma']


---
## Cell 13 — Run the Agent

> This cell makes live API calls. Typical runtime: **30–90 seconds** for 3 candidates.

In [69]:
final_report = run_screening(
    job_description=JOB_DESCRIPTION,
    resumes=RESUMES,
)

[1/3] Parsing job description...
      Done — Senior Machine Learning Engineer
[2/3] Evaluating 3 candidate(s)...
      Parsing   Alice Chen...
      Evaluating Alice Chen...
      Done:     Alice Chen
      Parsing   Bob Martinez...
      Evaluating Bob Martinez...
      Done:     Bob Martinez
      Parsing   Priya Sharma...
      Evaluating Priya Sharma...
      Done:     Priya Sharma
[3/3] Generating report...

  RESUME SCREENING REPORT — SENIOR MACHINE LEARNING ENGINEER
  Total Candidates Evaluated : 3
  Top Candidate              : Alice Chen (79.5/100)
  Recommended for Interview  : 2

RANKED CANDIDATES
----------------------------------------------------------------------
  #1  Alice Chen
      Overall Score   : 79.5/100  [Hire]
      Semantic Match  : 57.9%
      Skills Match    : 89.5%
      Experience      : 7.0 yrs — Senior
      Matched Skills  : Python, PyTorch or TensorFlow, MLflow or Kubeflow for experiment tracking and model registry, Docker and Kubernetes, AWS or GCP c

---
## Cell 14 — Inspect Individual Evaluations

Access per-candidate structured data after the run.

In [ ]:
evaluations = _state.get("evaluations", {})

if evaluations:
    # Display as a pandas DataFrame for easy comparison
    rows = []
    for ev in sorted(evaluations.values(), key=lambda x: x.overall_score, reverse=True):
        rows.append({
            "Candidate":          ev.candidate_name,
            "Overall Score":      f"{ev.overall_score:.1f}",
            "Semantic (%)": f"{ev.semantic_score*100:.1f}",
            "Skills Match (%)": f"{ev.skills_match_score*100:.1f}",
            "Experience (yrs)":   ev.years_experience,
            "Level":              ev.experience_level,
            "Recommendation":     ev.recommendation,
            "Matched Skills":     ", ".join(ev.matched_skills[:4]),
            "Skill Gaps":         ", ".join(ev.skill_gaps[:3]) or "—",
        })

    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    df.index.name = "Rank"

    print("\n── CANDIDATE COMPARISON TABLE ──")
    display(df)
else:
    print("No evaluations in state. Run Cell 13 first.")


── CANDIDATE COMPARISON TABLE ──


,Candidate,Overall Score,Semantic (%),Skills Match (%),Experience (yrs),Level,Recommendation,Matched Skills,Skill Gaps
Rank,,,,,,,,,
1,Alice Chen,79.5,57.9,89.5,7.0,Senior,Hire,"Python, PyTorch or TensorFlow, MLflow or Kubef...",—
2,Priya Sharma,77.0,56.4,84.2,5.0,Senior,Hire,"Python, PyTorch or TensorFlow, MLflow or Kubef...",—
3,Bob Martinez,54.5,69.8,47.4,3.0,Mid-Level,Maybe,"Python, Docker and Kubernetes, REST API develo...","PyTorch or TensorFlow, MLflow or Kubeflow for ..."


---
## Cell 15 — Use with Real PDF Resumes

Example of how to screen candidates from actual PDF files instead of pasted text.

In [94]:
# ── EXAMPLE: Screen from PDF files ───────────────────────────────────────────

pdf_resumes = [
    {"name": "Candidate A", "path": "/home/belal/Downloads/Belal_CV (4).pdf"},
    {"name": "Candidate B", "path": "/home/belal/Downloads/Belal_CV (3).pdf"},
    {"name": "Candidate C", "path": "/home/belal/Downloads/Belal_CV (2).pdf"},
]

# Extract text from each PDF before passing to the agent
for r in pdf_resumes:
    r["text"] = DocumentProcessor.extract(r["path"])
    print(f"Extracted {len(r['text'])} chars from {r['name']}")

# Now run screening
report = run_screening(JOB_DESCRIPTION, pdf_resumes)


Extracted 6005 chars from Candidate A
Extracted 7283 chars from Candidate B
Extracted 7283 chars from Candidate C
[1/3] Parsing job description...
      Done — Senior Machine Learning Engineer
[2/3] Evaluating 3 candidate(s)...
      Parsing   Candidate A...
      Evaluating Belal Fathy Abdelf attah...
      Done:     Belal Fathy Abdelf attah
      Parsing   Candidate B...
      Evaluating Belal Fathy Abdelf attah...
      Done:     Belal Fathy Abdelf attah
      Parsing   Candidate C...
      Evaluating Belal Fathy Abdelf attah...
      Done:     Belal Fathy Abdelf attah
[3/3] Generating report...

  RESUME SCREENING REPORT — SENIOR MACHINE LEARNING ENGINEER
  Total Candidates Evaluated : 1
  Top Candidate              : Belal Fathy Abdelf attah (24.2/100)
  Recommended for Interview  : 0

RANKED CANDIDATES
----------------------------------------------------------------------
  #1  Belal Fathy Abdelf attah
      Overall Score   : 24.2/100  [No Hire]
      Semantic Match  : 45.0%
    